# Small Language Model (SLM) from Scratch
### A GPT-style transformer trained on TinyStories

This notebook trains a small language model from scratch using PyTorch.  
Architecture: GPT-style decoder-only transformer with causal self-attention.  
Dataset: [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) — short children's stories, ideal for small models.

**Runtime:** Set to `GPU → T4` via *Runtime → Change runtime type* before running.

## 1. Install dependencies

In [ ]:
%pip install -q torch tiktoken datasets tqdm pydantic matplotlib numpy

## 2. GPU check & Google Drive mount
Checkpoints are saved to Google Drive so they persist after the session ends.

In [1]:
import torch

# ── GPU check ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}  |  VRAM: {mem:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow on CPU.")
    print("Go to Runtime → Change runtime type → GPU → T4")

Go to Runtime → Change runtime type → GPU → T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All checkpoints and outputs save here — survives session restarts
DRIVE_DIR = "/content/drive/MyDrive/slm_from_scratch"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("plots", exist_ok=True)

print(f"Drive directory: {DRIVE_DIR}")

Mounted at /content/drive
Drive directory: /content/drive/MyDrive/slm_from_scratch


## 3. Configuration
Two config classes control the full experiment:
- `GPTConfig` — model architecture (layers, heads, embedding size)
- `TrainConfig` — training schedule (LR, batch size, iterations)

**Colab T4 (~15 GB VRAM) tip:** The defaults below are tuned to fit comfortably.  
If you hit OOM, reduce `batch_size` to 16 or `n_embd` to 256.

In [3]:
from pydantic import BaseModel

class GPTConfig(BaseModel):
    block_size: int = 128      # context window length
    vocab_size: int = 50304    # GPT-2 tokenizer vocab size (padded to multiple of 64)
    n_layer: int = 6           # number of transformer blocks
    n_head: int = 6            # number of attention heads
    n_embd: int = 384          # embedding dimension (must be divisible by n_head)
    dropout: float = 0.1
    bias: bool = True


class TrainConfig(BaseModel):
    # Optimiser
    learning_rate: float = 1e-4
    min_lr: float = 5e-5
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95

    # Schedule
    max_iters: int = 20000
    warmup_steps: int = 1000
    eval_iters: int = 500         # batches used to estimate loss at each eval
    eval_interval: int = 500      # evaluate every N steps

    # Batch
    batch_size: int = 32
    block_size: int = 128
    gradient_accumulation_steps: int = 32

    # Reproducibility
    seed: int = 42


# ── Instantiate configs ────────────────────────────────────────────────────
model_config = GPTConfig()
train_config = TrainConfig()

print("Model config:", model_config.model_dump())
print("\nTrain config:", train_config.model_dump())

Model config: {'block_size': 128, 'vocab_size': 50304, 'n_layer': 6, 'n_head': 6, 'n_embd': 384, 'dropout': 0.1, 'bias': True}

Train config: {'learning_rate': 0.0001, 'min_lr': 5e-05, 'weight_decay': 0.1, 'beta1': 0.9, 'beta2': 0.95, 'max_iters': 20000, 'warmup_steps': 1000, 'eval_iters': 500, 'eval_interval': 500, 'batch_size': 32, 'block_size': 128, 'gradient_accumulation_steps': 32, 'seed': 42}


## 4. Model architecture

The model is a decoder-only transformer (GPT-style) with:
- **Token + positional embeddings** — learned, not sinusoidal
- **Causal self-attention** — each token can only attend to past tokens
- **Pre-norm residual blocks** — LayerNorm before attention and MLP (more stable than post-norm)
- **Weight tying** — embedding and LM head share weights (reduces parameters)
- **Flash attention** — uses PyTorch's `scaled_dot_product_attention` when available (faster + less memory)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ── Layer Norm (with optional bias) ───────────────────────────────────────
class LayerNorm(nn.Module):
    """LayerNorm with optional bias (PyTorch built-in doesn't support bias=False)."""
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)


# ── Causal Self-Attention ─────────────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    """
    Multi-head causal (masked) self-attention.

    Uses Flash Attention (F.scaled_dot_product_attention) when available
    for better memory efficiency and speed. Falls back to manual attention
    with a causal mask otherwise.
    """
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0

        # Single linear projects Q, K, V all at once
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        self.n_head = config.n_head
        self.n_embd = config.n_embd

        # Use Flash Attention if PyTorch >= 2.0
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            # Fallback: register causal mask as a buffer
            self.register_buffer(
                "bias",
                torch.tril(torch.ones(config.block_size, config.block_size))
                    .view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x):
        B, T, C = x.size()  # batch, sequence length, embedding dim

        # Compute Q, K, V in one matmul then split
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(
                q, k, v,
                is_causal=True,
                dropout_p=self.attn_dropout.p if self.training else 0.0
            )
        else:
            att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y


# ── MLP (Feed-Forward Network) ────────────────────────────────────────────
class MLP(nn.Module):
    """
    Position-wise feed-forward network.
    Expands embedding by 4x, applies GELU, projects back.
    """
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu   = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))


# ── Transformer Block ─────────────────────────────────────────────────────
class Block(nn.Module):
    """
    One transformer block: pre-norm attention + pre-norm MLP with residual connections.
    Pre-norm (LN before sublayer) is more stable than post-norm during training.
    """
    def __init__(self, config):
        super().__init__()
        self.ln1  = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2  = LayerNorm(config.n_embd, config.bias)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))  # residual + attention
        x = x + self.mlp(self.ln2(x))   # residual + MLP
        return x


# ── Full GPT Model ────────────────────────────────────────────────────────
class GPT(nn.Module):
    """
    GPT-style decoder-only transformer.

    Key design choices:
    - Learned token + positional embeddings
    - Stack of N transformer blocks
    - Weight tying: embedding matrix == LM head (saves ~20M params at this scale)
    - Standard weight initialisation (std=0.02, zeros for biases)
    """
    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(config.vocab_size, config.n_embd),   # token embeddings
            wpe  = nn.Embedding(config.block_size, config.n_embd),   # position embeddings
            drop = nn.Dropout(config.dropout),
            h    = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, config.bias),            # final layer norm
        ))
        # self.lm_head = nn.Linear(config.vocab_size, config.n_embd, bias=False)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: token embedding and LM head share the same matrix
        self.transformer.wte.weight = self.lm_head.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        pos = torch.arange(0, T, device=idx.device)

        tok_emb = self.transformer.wte(idx)   # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)   # (T, n_embd)

        x = self.transformer.drop(tok_emb + pos_emb)

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)   # (B, T, vocab_size)


        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Autoregressively generate tokens given a prompt.
        idx: (B, T) tensor of token ids
        """
        for _ in range(max_new_tokens):
            # Crop to block_size if sequence gets too long
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)

        return idx


# ── Parameter count ───────────────────────────────────────────────────────
model = GPT(model_config)

def format_num(n):
    if n >= 1e9: return f"{n/1e9:.2f}B"
    if n >= 1e6: return f"{n/1e6:.2f}M"
    if n >= 1e3: return f"{n/1e3:.2f}K"
    return str(n)

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {format_num(total)}")
print(f"Trainable parameters: {format_num(trainable)}")

Total parameters:     30.01M
Trainable parameters: 30.01M


## 5. Data pipeline

Downloads and tokenizes the **TinyStories** dataset using the GPT-2 tokenizer (`tiktoken`).  
Tokenized data is stored as memory-mapped `.bin` files — this lets the data loader read random batches without loading everything into RAM.

The binary files are also copied to Google Drive so you don't need to re-download on session restart.

In [5]:
DATA_DIR = "data"
TOKENIZER_NAME = "gpt2"

In [6]:
import os
import numpy as np
from tqdm.auto import tqdm
import tiktoken
from datasets import load_dataset


def load_data(subset: bool = False):
    """
    Load TinyStories from HuggingFace.
    Set subset=True for a quick smoke-test (100 train / 20 val examples).
    """
    if subset:
        return load_dataset(
            "roneneldan/TinyStories",
            split={"train": "train[:100]", "validation": "validation[:20]"}
        )
    return load_dataset("roneneldan/TinyStories")


def tokenize_dataset(ds, num_proc: int = 2):
    """Encode every story with tiktoken and return token ids + lengths."""
    enc = tiktoken.get_encoding(TOKENIZER_NAME)

    def process(example):
        ids = enc.encode_ordinary(example["text"])
        return {"ids": ids, "len": len(ids)}

    return ds.map(
        process,
        remove_columns=["text"],
        desc="Tokenizing",
        num_proc=num_proc,
    )


def write_bin_file(dset, split: str, data_dir: str = DATA_DIR):
    """
    Write tokenized split to a flat uint16 binary file via memory-mapping.
    Using memmap avoids materialising the full array in RAM.
    """
    os.makedirs(data_dir, exist_ok=True)
    filename = os.path.join(data_dir, f"{split}.bin")

    arr_len = np.sum(dset["len"], dtype=np.uint64)
    arr = np.memmap(filename, dtype=np.uint16, mode="w+", shape=(arr_len,))

    total_batches = min(1024, len(dset))
    idx = 0

    for batch_idx in tqdm(range(total_batches), desc=f"Writing {split}.bin"):
        batch = (
            dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True)
            .with_format("numpy")
        )
        arr_batch = np.concatenate(batch["ids"])
        arr[idx: idx + len(arr_batch)] = arr_batch
        idx += len(arr_batch)

    arr.flush()
    print(f"  Saved {filename}  ({arr_len:,} tokens)")


def run_pipeline(subset: bool = False, num_proc: int = 2):
    """Full pipeline: download → tokenize → write .bin files."""
    print("Loading dataset...")
    ds = load_data(subset=subset)

    print("Tokenizing...")
    tokenized = tokenize_dataset(ds, num_proc=num_proc)

    print("Writing binary files...")
    for split, dset in tokenized.items():
        write_bin_file(dset, split)

    print("\nDone ✅")

In [ ]:
# ── Run the pipeline ──────────────────────────────────────────────────────
# Set subset=False to use the full TinyStories dataset (~2.1M stories)
# Set subset=True for a quick test run (finishes in seconds)

run_pipeline(subset=False, num_proc=2)

# ── Back up .bin files to Drive ───────────────────────────────────────────
import shutil
for split in ["train", "validation"]:
    src = f"data/{split}.bin"
    dst = f"{DRIVE_DIR}/{split}.bin"
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Backed up {src} → {dst}")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

...

Writing binary files...


Writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

  Saved data/train.bin  (471,872,517 tokens)


Writing validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

  Saved data/validation.bin  (4,743,928 tokens)

Done ✅
Backed up data/train.bin → /content/drive/MyDrive/slm_from_scratch/train.bin
Backed up data/validation.bin → /content/drive/MyDrive/slm_from_scratch/validation.bin


## 6. Batch loader

Reads random batches from the memory-mapped `.bin` files.  
`x` = input tokens, `y` = the same tokens shifted by 1 (next-token prediction target).  
Using `.pin_memory()` speeds up CPU→GPU transfer.

In [7]:
def get_batch(split, model_config, train_config, device):
    """
    Sample a random batch from the pre-tokenized binary file.

    Returns:
        x: (batch_size, block_size) — input token ids
        y: (batch_size, block_size) — target token ids (x shifted right by 1)
    """
    filename = "train.bin" if split == "train" else "validation.bin"
    filepath = os.path.join(DATA_DIR, filename)

    # If files don't exist (e.g. session restart), re-run pipeline
    if not os.path.exists(filepath):
        # Try restoring from Drive first
        drive_path = f"{DRIVE_DIR}/{filename}"
        if os.path.exists(drive_path):
            shutil.copy2(drive_path, filepath)
            print(f"Restored {filepath} from Drive")
        else:
            print(f"{filepath} not found — re-running pipeline...")
            run_pipeline(subset=False, num_proc=2)

    data = np.memmap(filepath, dtype=np.uint16, mode="r")

    ix = torch.randint(len(data) - model_config.block_size, (train_config.batch_size,))

    x = torch.stack([torch.from_numpy(data[i     : i +     model_config.block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1 : i + 1 + model_config.block_size].astype(np.int64)) for i in ix])

    if device.type == "cuda":
        x = x.pin_memory().to(device, non_blocking=True)
        y = y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)

    return x, y

## 7. Trainer

Key training techniques used:
- **Mixed precision (bfloat16 / float16)** — halves memory use, speeds up training on modern GPUs
- **Gradient accumulation** — simulates a larger effective batch size without needing more VRAM
- **Gradient clipping** — caps gradient norm at 0.5 to prevent exploding gradients
- **Warmup + cosine LR decay** — linear warmup for `warmup_steps`, then cosine decay to `min_lr`
- **Best model checkpointing** — saves to Google Drive whenever validation loss improves

In [8]:
from contextlib import nullcontext
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
import matplotlib.pyplot as plt


def setup_device():
    """Detect device, set AMP dtype and autocast context."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    ptdtype = (
        torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported()
        else torch.float32 if device.type == "cpu"
        else torch.float16
    )

    ctx = (
        nullcontext() if device.type == "cpu"
        else torch.amp.autocast(device_type=device.type, dtype=ptdtype)
    )

    return device, ctx, ptdtype


def build_optimizer(model, config):
    return torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        betas=(config.beta1, config.beta2),
        weight_decay=config.weight_decay,
        eps=1e-9,
    )


def build_scheduler(optimizer, config):
    warmup = LinearLR(optimizer, total_iters=config.warmup_steps)
    decay  = CosineAnnealingLR(
        optimizer,
        T_max=config.max_iters - config.warmup_steps,
        eta_min=config.min_lr,
    )
    return SequentialLR(optimizer, schedulers=[warmup, decay], milestones=[config.warmup_steps])


@torch.no_grad()
def estimate_loss(model, model_config, train_config, ctx, device):
    """Estimate mean loss over eval_iters batches for train and val splits."""
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(train_config.eval_iters)
        for k in range(train_config.eval_iters):
            X, Y = get_batch(split, model_config, train_config, device)
            with ctx:
                _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


def train(model, model_config, train_config, drive_dir=DRIVE_DIR):
    """
    Full training loop with:
    - gradient accumulation
    - mixed precision (AMP)
    - gradient clipping
    - warmup + cosine LR schedule
    - best-model checkpointing to Google Drive
    """
    device, ctx, ptdtype = setup_device()
    print(f"Training on: {device}  |  dtype: {ptdtype}")

    torch.manual_seed(train_config.seed)
    model.to(device)

    total_params    = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters — total: {format_num(total_params)}  |  trainable: {format_num(trainable_params)}")

    optimizer = build_optimizer(model, train_config)
    scheduler = build_scheduler(optimizer, train_config)
    scaler    = torch.amp.GradScaler(enabled=(device.type == "cuda" and ptdtype == torch.float16))

    best_val_loss  = float("inf")
    checkpoint_path = os.path.join(drive_dir, "best_model.pt")

    train_losses, val_losses = [], []

    for iter_num in range(train_config.max_iters):

        # ── Evaluation ────────────────────────────────────────────────────
        if iter_num % train_config.eval_interval == 0 and iter_num > 0:
            losses = estimate_loss(model, model_config, train_config, ctx, device)
            lr_now = optimizer.param_groups[0]["lr"]

            print(
                f"Step {iter_num:>6}  |  "
                f"train {losses['train']:.4f}  |  "
                f"val {losses['val']:.4f}  |  "
                f"lr {lr_now:.2e}"
            )

            train_losses.append(losses["train"])
            val_losses.append(losses["val"])

            # Save best model to Drive
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save({
                    "model":     model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "iter":      iter_num,
                    "val_loss":  best_val_loss,
                    "model_config": model_config.model_dump(),
                    "train_config": train_config.model_dump(),
                }, checkpoint_path)
                print(f"  ✅ Saved best model (val_loss={best_val_loss:.4f}) → {checkpoint_path}")

        # ── Gradient accumulation loop ────────────────────────────────────
        optimizer.zero_grad(set_to_none=True)

        for micro_step in range(train_config.gradient_accumulation_steps):
            X, Y = get_batch("train", model_config, train_config, device)
            print(f"  Micro-step {micro_step}: X shape={X.shape}, Y shape={Y.shape}")
            with ctx:
                _, loss = model(X, Y)
                loss = loss / train_config.gradient_accumulation_steps
            scaler.scale(loss).backward()

        # ── Gradient clipping ─────────────────────────────────────────────
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)

        # ── Optimiser + scheduler step ────────────────────────────────────
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

    print("\nTraining complete ✅")
    print(f"Best validation loss: {best_val_loss:.4f}")

    return {
        "train_losses": train_losses,
        "val_losses":   val_losses,
        "best_val_loss": best_val_loss,
    }

## 8. Run training

This will take ~2–4 hours on a Colab T4 GPU for the full dataset at 20,000 steps.  
The best checkpoint is saved to your Google Drive automatically whenever validation loss improves.  

> **Tip:** If your Colab session disconnects, reload the notebook, re-run cells 1–7, then load the checkpoint from Drive using the cell in Section 10.

In [ ]:

# ── Re-initialise model ───────────────────────────────────────────────────
model = GPT(model_config)

# ── Train ─────────────────────────────────────────────────────────────────
results = train(model, model_config, train_config)

Streaming output truncated to the last 5000 lines.
  Micro-step 8: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 9: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 10: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 11: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 12: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 13: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 14: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 15: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 16: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 17: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 18: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 19: X shape=torch.Size([32, 128]), Y shape=torch.Size([32, 128])
  Micro-step 20

## 9. Training curves

Plot training and validation loss over time.  
Saved to `plots/loss_curve.png` and to your Google Drive.

In [10]:
%matplotlib inline

def plot_losses(train_losses, val_losses, eval_interval,
                save_path="plots/loss_curve.png", drive_dir=DRIVE_DIR):
    steps = [i * eval_interval for i in range(1, len(train_losses) + 1)]

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(steps, train_losses, label="Train loss",      linewidth=2)
    ax.plot(steps, val_losses,   label="Validation loss", linewidth=2, linestyle="--")

    ax.set_xlabel("Training steps")
    ax.set_ylabel("Cross-entropy loss")
    ax.set_title("SLM training — TinyStories")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)

    # Back up to Drive
    shutil.copy2(save_path, os.path.join(drive_dir, "loss_curve.png"))
    print(f"Saved → {save_path}  |  backed up to Drive")

    plt.show()
    plt.close()


plot_losses(results["train_losses"], results["val_losses"], train_config.eval_interval)

NameError: name 'results' is not defined

## 10. Resume from checkpoint *(run this if session restarted)*

If your Colab session disconnected, use this cell to restore the best model from Drive before generating.

In [15]:
def load_checkpoint(drive_dir=DRIVE_DIR):
    """Load best model checkpoint from Google Drive."""
    checkpoint_path = os.path.join(drive_dir, "best_model.pt")

    if not os.path.exists(checkpoint_path):
        print(f"No checkpoint found at {checkpoint_path}")
        return None

    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    print(f"Loaded checkpoint from step {checkpoint['iter']}  | val_loss={checkpoint['val_loss']:.4f}")

    restored_model_config = GPTConfig(**checkpoint["model_config"])
    restored_model = GPT(restored_model_config)
    restored_model.load_state_dict(checkpoint["model"])

    return restored_model

# Uncomment to restore after a session restart:
# model = load_checkpoint()

Loaded checkpoint from step 1000  | val_loss=2.8619


## 11. Text generation

Generate text from a prompt using the trained model.  

Parameters:
- `temperature` — higher = more random, lower = more deterministic (try 0.7–1.0)
- `top_k` — only sample from the top-k most likely tokens at each step (try 40–100)
- `max_new_tokens` — how many tokens to generate

In [11]:
import tiktoken

def generate_text(model, prompt, max_new_tokens=200, temperature=0.8, top_k=50, device=None):
    """
    Generate text continuation from a prompt.

    Args:
        model: trained GPT model
        prompt: string to condition generation on
        max_new_tokens: number of tokens to generate
        temperature: sampling temperature (higher = more creative)
        top_k: top-k filtering (None = no filtering)
        device: torch device (auto-detected if None)
    """
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)
    model.eval()

    enc = tiktoken.get_encoding("gpt2")
    input_ids = enc.encode(prompt)
    x = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        output_ids = model.generate(
            x,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )

    return enc.decode(output_ids[0].tolist())


# ── Try it out ────────────────────────────────────────────────────────────
prompts = [
    "Once upon a time there was a little girl named",
    "The dog ran across the field and",
    "Tom and his best friend decided to",
]

for prompt in prompts:
    print(f"Prompt: {prompt!r}")
    print("-" * 60)
    output = generate_text(model, prompt, max_new_tokens=150, temperature=0.8, top_k=50)
    print(output)
    print("\n" + "=" * 60 + "\n")

Prompt: 'Once upon a time there was a little girl named'
------------------------------------------------------------
Once upon a time there was a little girl named Sue. She loved to play outside in the garden. One day, Lily saw a big tree. It was the lake and wanted to go to the tree. Sarah wanted to jump through the tree, but the tree was too fast. She searched, but it couldn't get closer. She fell down and away.

Lily felt sad and scared. The bird was sad and sad. She ran away to her mom and said, "I'm sorry, little rabbit," she said, "I'm sorry, Lily. I didn't want to touch it." 

On the end of the park, Lily got scared of the ball. She went home and felt sad. She said, "Don't worry, Lily. I promise


Prompt: 'The dog ran across the field and'
------------------------------------------------------------
The dog ran across the field and began to jump in the sky. He was so happy he had seen any way to get some water. He ran to the park until he reached a closer closer pond. He was so

## Done!

Your SLM is trained and generating text. The best checkpoint is saved at:
`Google Drive → My Drive → slm_from_scratch → best_model.pt`

**Next steps:**
- Try different prompts and temperatures in Section 11
- Experiment with the config (more layers, larger embedding) in Section 3
- Use `subset=True` in the data pipeline for quick architecture experiments